In [ ]:
import pandas as pd
import altair as alt
from datetime import datetime
from natsort import natsorted

In [ ]:
data = {'BARD1': '../Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'}

alt.data_transformers.disable_max_rows()

In [ ]:
def get_gmm_threshold(df): #estimates thresholds by finding highest scoring/lowest scoring functionally abnormal/functionally normal variants respectively
    
    ab_df = df.loc[df['functional_consequence'].isin(['functionally_abnormal'])]
    norm_df = df.loc[df['functional_consequence'].isin(['functionally_normal'])]

    # now we get the scores that are the closest to the (n)ormal and (a)bnormal thresholds
    upprthresh = norm_df['score'].min()
    lwrthresh = ab_df['score'].max()


    thresholds = [lwrthresh, upprthresh]

    return thresholds

In [ ]:
def read_scores(file): #reads score from excel file
    df = pd.read_excel(file, sheet_name = 'scores')
    df = df.loc[~df['variant_qc_flag'].isin(['WARN'])]
    threshold_df = pd.read_excel(file, sheet_name = 'thresholds')
    df = df.rename(columns = {'consequence': 'Consequence'}) #Comment out when column name changes back to 'Consequence'
    
    thresholds = get_gmm_threshold(df)


    df.loc[df['Consequence'].str.contains('missense'), 'Consequence'] = 'Missense'
    df.loc[df['Consequence'] == 'synonymous_variant', 'Consequence'] = 'Synonymous'
    df.loc[df['Consequence'] == 'intron_variant', 'Consequence'] = 'Intron'
    df.loc[df['Consequence'] == 'stop_gained', 'Consequence'] = 'Stop Gained'
    df.loc[df['Consequence'] == 'stop_lost', 'Consequence'] = 'Stop Lost'
    df.loc[df['Consequence'].str.contains('site'), 'Consequence'] = 'Canonical Splice'
    df.loc[df['Consequence'].str.contains('ing_var'), 'Consequence'] = 'Splice Region'
    df.loc[df['Consequence'].str.contains('UTR'), 'Consequence'] = 'UTR Variant'
    df.loc[df['Consequence'] == 'start_lost', 'Consequence'] = 'Start Lost'

    df.loc[df['var_type'] == '3bp_del', 'Consequence'] = '3bp Deletion'
    df.loc[df['var_type'] == 'snv', 'var_type'] = 'SNV'
    df.loc[df['var_type'] == '3bp_del', 'var_type'] = 'Deletion'

    df['exon'] = df['exon'].transform(lambda x: 'Exon ' + x.split('_')[1][1::])
    
    return df, thresholds

In [ ]:
def make_plot(df, gene, sge_threshold):

    chromosome = df['chrom'][0]

    exon_order = natsorted(list(set(df['exon'].tolist())))
    
    palette = [
    '#006616', # dark green,
    '#81B4C7', # dusty blue
    '#ffcd3a', # yellow
    '#6AA84F', # med green
    '#93C47D', # light green
    '#888888', # med gray
    '#000000', # black
    '#1170AA', # darker blue
    '#CFCFCF', # light gray
    '#FF9A00'   #orange
        
    ]
    
    
    variant_types = [
        'Synonymous',
        'Missense',  
        'Stop Gained',
        'Intron', 
        'UTR Variant',
        'Stop Lost',
        'Start Lost',
        'Canonical Splice', 
        'Splice Region',
        '3bp Deletion'
    ]

    
    chart = alt.Chart(df).mark_point().encode(
        x = alt.X('pos:Q',
                  title = f' Genomic Coordinate ({chromosome})',
                  axis = alt.Axis(titleFontSize = 18,
                                 labelFontSize = 16),
                  scale = alt.Scale(zero = False)
                 ),
        y = alt.Y('score:Q',
                  title = 'Fitness Score',
                  axis = alt.Axis(titleFontSize = 18,
                                 labelFontSize = 16)
                 ),
        color = alt.Color('Consequence:N',
                          scale = alt.Scale(
                              domain = variant_types,
                              range = palette
                          ),
                          legend = alt.Legend(
                              symbolStrokeWidth = 2,
                              titleFontSize = 18,
                              labelFontSize = 16
                          )
                         ),
        shape = alt.Shape('var_type:N',
                          legend = alt.Legend(
                              title = 'Variant Type',
                              symbolStrokeColor = 'black',
                              symbolStrokeWidth = 2,
                              symbolFillColor = 'black',
                               titleFontSize = 18,
                              labelFontSize = 16
                          )
                         )
    ).properties(
        width = 600,
        height = 150
    )

    nf_line = alt.Chart(df).mark_rule(
        color='black', strokeDash=[8,8], strokeWidth=1
    ).encode(
        y=alt.datum(sge_threshold[0])
    )
    
    func_line = alt.Chart(df).mark_rule(
        color='#888888', strokeDash=[8,8], strokeWidth=1
    ).encode(
        y=alt.datum(sge_threshold[1])
    )


    chart = alt.layer(chart, nf_line, func_line)

    chart = chart.facet(
        facet = alt.Facet(
            'exon:N',
            sort = exon_order,
            title = gene
        ),
        columns = 2
    ).resolve_scale(
        x = 'independent'
    ).configure_header(
        labelFontSize = 16,
        titleFontSize = 18
    )

    chart.display()

    return chart

In [ ]:
def main(save = False):

    genes = list(data.keys())

    for gene in genes:
        data_path = data[gene]
        df, thresholds = read_scores(data_path)

        final_plot = make_plot(df,gene, thresholds)

        if save:
            date = datetime.now().strftime('%Y%m%d')
            save_string = f'/Users/ivan/Desktop/BARD1_draft_figs/supp_figs/{date}_scores_acrossexons.svg'

            final_plot.save(save_string)

In [ ]:
main(save = False)